# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/ppo/#ppo_atari_envpoolpy

In [1]:
# @launchit.collected

In [2]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter # @launchit.collect
import random
import datetime
import json
import pprint
import re
import uuid
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import Dataset, DataLoader
from torch.distributions.categorical import Categorical

import envpool
import gymnasium as gym
import ale_py
import av

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect

from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter
from logging_utils import *
from artifact_registry import *
from torch_helpers import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

# Init

In [3]:
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    private_data_path=None,
    run_path=None,
    self_fname=None,
    self_name=None,
    subproject_name=None,
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG = CONFIG._replace(private_data_path=os.path.join(CONFIG.data_path, CONFIG.subproject_name))

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17a_ppo_atari_envpool_01.ipynb',
 'self_name': '17a_ppo_atari_envpool_01',
 'subproject_name': '17_rl',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [4]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    # System params
    random_seed: int = None
    torch_deterministic: bool = True 

    # Environment params
    env_id: str = None
    envs_count: int = 8 # the number of parallel game environments
    test_env_id: str = None

    # Agent params
    separate_networks: bool = None # where Actor / Critic have separate networks or share share the same network

    # Training params
    rollouts_count: int = 10 
    steps_count: int = 128 # the number of steps to run in each environment per policy rollout
    minibatches_count: int = 4
    update_epochs: int = 4 # the K epochs to update the policy
    optimizer: str = 'Adam(eps=0.00001)'
    learn_rate: str = '0.00025,linear()'
    capture_video: str = None # video capture policy depending on rollouts, e.g. every(200)

    # RL params
    gamma: float = 0.99 # the discount factor gamma
    gae_lambda: float = 0.95 # the lambda for the general advantage estimation
    
    # PPO related
    clip_coef: float = 0.1 # the surrogate clipping coefficient
    clip_vloss: bool = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    ent_coef: float = 0.01 # coefficient of the entropy
    vf_coef: float = 0.5 # coefficient of the value function
    max_grad_norm: float = 0.5 # the maximum norm for the gradient clipping
    target_kl: float = None # e target KL divergence threshold
    norm_adv: bool = True # Toggles advantages normalization

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()
HP.random_seed = 42

# Launch

## LaunchState

In [5]:
@dataclass(slots=True)
class LaunchState:
    envs: list = None
    agent: object = None

    @staticmethod
    def create_capture_video_policy(capture_video):
        def inner_create_capture_video_policy(policy_name, every_rollouts_count):
            if policy_name == 'every':
                def every_thunk(rollout_serial):
                    return (rollout_serial % every_rollouts_count) == 0

                return every_thunk
            
            assert False, f'Unsupported capture_video="{module_name}"'
                
        ump = hp_parse_universal_module(capture_video)
        return inner_create_capture_video_policy(ump.module_name, *ump.args, **ump.kwargs)

LS = LaunchState()

## new_artifact_registry

In [6]:
def new_artifact_registry(is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ArtifactRegistry(CONFIG.model_group_uri)

## new_summary_writer

In [7]:
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Bootstrap

In [8]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', HP.launch_id)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    random.seed(HP.random_seed)
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if HP.torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()
artifact_registry = new_artifact_registry()
artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    
meta = dict(
    optuna_trial_number=getattr(optuna_trial, 'number', None),
    hypers=HP._asdict(), 
    config=CONFIG._asdict(), 
)

with io.StringIO() as b:
    json.dump(meta, b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = lc.name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

Random seed=42
torch.backends.cudnn.deterministic=True
Tensorboard run=17a_ppo_atari_envpool_01/0


<Mock name='mock.add_text()' id='140053534777920'>

# Environment

## Configure 

In [9]:
# @launchit.disable
# @launchit.collect
HP.env_id = 'Breakout-v5'
HP.envs_count = 8 # the number of parallel game environments
HP.test_env_id = 'BreakoutNoFrameskip-v4'
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'Breakout-v5',
 'envs_count': 8,
 'test_env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': None,
 'rollouts_count': 10,
 'steps_count': 128,
 'minibatches_count': 4,
 'update_epochs': 4,
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()',
 'capture_video': None,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True}


## RecordEpisodeStatistics

In [10]:
class RecordEpisodeStatistics(gym.Wrapper):
    def __init__(self, env, deque_size=100):
        super().__init__(env)
        self.num_envs = getattr(env, "num_envs", 1)
        self.episode_returns = None
        self.episode_lengths = None

    def reset(self, **kwargs):
        # observations = super().reset(**kwargs)
        observations = self.env.reset()
        self.episode_returns = np.zeros(self.num_envs, dtype=np.float32)
        self.episode_lengths = np.zeros(self.num_envs, dtype=np.int32)
        self.lives = np.zeros(self.num_envs, dtype=np.int32)
        self.returned_episode_returns = np.zeros(self.num_envs, dtype=np.float32)
        self.returned_episode_lengths = np.zeros(self.num_envs, dtype=np.int32)
        return observations

    def step(self, action):
        observations, rewards, terminated, truncated, infos = super().step(action)
        self.episode_returns += infos["reward"]
        self.episode_lengths += 1
        self.returned_episode_returns[:] = self.episode_returns
        self.returned_episode_lengths[:] = self.episode_lengths
        self.episode_returns *= 1 - infos["terminated"]
        self.episode_lengths *= 1 - infos["terminated"]
        infos["r"] = self.returned_episode_returns
        infos["l"] = self.returned_episode_lengths
        return (
            observations,
            rewards,
            terminated,
            truncated,
            infos,
        )

## create_envpool

In [11]:
def create_envpool(envs_count):
    envs = envpool.make(
        HP.env_id,
        env_type='gym',
        num_envs=envs_count,
        episodic_life=True,
        reward_clip=True,
        seed=HP.random_seed,
    )
    envs = RecordEpisodeStatistics(envs)
    envs.num_envs = envs_count
    envs.single_action_space = envs.action_space
    envs.single_observation_space = envs.observation_space
    assert isinstance(envs.action_space, gym.spaces.Discrete), "only discrete action space is supported"
    return envs

## create_test_env

In [12]:
def create_test_env(env_id, video_dir_name=None):
    if video_dir_name is not None:
        env = gym.make(env_id, render_mode='rgb_array')
        env = gym.wrappers.RecordVideo(env, video_dir_name, episode_trigger=lambda episode_id: episode_id == 0)
    else:
        env = gym.make(env_id)
        
    env = gym.wrappers.RecordEpisodeStatistics(env)
    env = NoopResetEnv(env, noop_max=30)
    env = MaxAndSkipEnv(env, skip=4)
    env = EpisodicLifeEnv(env) 
    
    if 'FIRE' in env.unwrapped.get_action_meanings():
        env = FireResetEnv(env)
        
    env = ClipRewardEnv(env)
    # Following three directly influence envs.single_observation_space
    env = gym.wrappers.ResizeObservation(env, (84, 84))
    env = gym.wrappers.GrayscaleObservation(env)
    env = gym.wrappers.FrameStackObservation(env, 4)
    return env

## Create

In [13]:
LS.envs = create_envpool(HP.envs_count)
LOG(f'{LS.envs.single_observation_space=}, {LS.envs.single_action_space=}, {LS.envs.metadata=}')

LS.envs.single_observation_space=Box(0, 255, (4, 84, 84), uint8), LS.envs.single_action_space=Discrete(4), LS.envs.metadata={'render_modes': ['rgb_array', 'human']}


In [14]:
# LS.envs.reset()
# LS.envs.step(np.array([0] * HP.envs_count))

# Agent

## Configure

In [15]:
# @launchit.disable
# @launchit.collect
HP.separate_networks = False
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'Breakout-v5',
 'envs_count': 8,
 'test_env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': False,
 'rollouts_count': 10,
 'steps_count': 128,
 'minibatches_count': 4,
 'update_epochs': 4,
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()',
 'capture_video': None,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True}


## Agent

In [16]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = 512
        actions_count: int = None 
        separate_networks: bool = None
    
    def __init__(self, params):
        super().__init__()
        self.params = params

        if params.separate_networks:
            self.actor = nn.Sequential(
                self._layer_init(nn.Conv2d(4, 32, 8, stride=4)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                self._layer_init(nn.Linear(64 * 7 * 7, params.d_model)),
                nn.ReLU(),
                self._layer_init(nn.Linear(params.d_model, params.actions_count), std=0.01)
            )

            self.critic = nn.Sequential(
                self._layer_init(nn.Conv2d(4, 32, 8, stride=4)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                self._layer_init(nn.Linear(64 * 7 * 7, params.d_model)),
                nn.ReLU(),
                self._layer_init(nn.Linear(params.d_model, 1), std=1)
            )
        else:
            self.network = nn.Sequential(
                self._layer_init(nn.Conv2d(4, 32, 8, stride=4)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                self._layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                self._layer_init(nn.Linear(64 * 7 * 7, params.d_model)),
                nn.ReLU(),
            )
            self.actor = self._layer_init(nn.Linear(params.d_model, params.actions_count), std=0.01)
            self.critic = self._layer_init(nn.Linear(params.d_model, 1), std=1)

    def get_value(self, x):
        if self.params.separate_networks:
            return self.critic(x / 255.0)
        else:
            hidden = self.network(x / 255.0)
            return self.critic(hidden)

    ForwardResult = namedtuple('ForwardResult', 'action, log_action_probs, probs_entropy, value')

    def forward(self, x, action=None): # get_action_and_value
        if self.params.separate_networks:
            x = x / 255.0
            logits = self.actor(x) # [batch, actions] 
            probs = Categorical(logits=logits) # [batch]
            action = lu.coalesce(action, lambda: probs.sample()) # [batch]
            return Agent.ForwardResult(
                action=action, 
                log_action_probs=probs.log_prob(action), 
                probs_entropy=probs.entropy(), 
                value=self.critic(x), # [batch, 1]
            )
        else:
            hidden = self.network(x / 255.0) # [batch, d_model] 
            logits = self.actor(hidden) # [batch, actions] 
            probs = Categorical(logits=logits) # [batch]
            action = lu.coalesce(action, lambda: probs.sample()) # [batch]
            return Agent.ForwardResult(
                action=action, 
                log_action_probs=probs.log_prob(action), 
                probs_entropy=probs.entropy(), 
                value=self.critic(hidden), # [batch, 1]
            )

    @staticmethod
    def _layer_init(layer, std=np.sqrt(2), bias_const=0.0):
        torch.nn.init.orthogonal_(layer.weight, std)
        torch.nn.init.constant_(layer.bias, bias_const)
        return layer

## Smoke test

In [17]:
# @launchit.disable
ap = Agent.Params(
    actions_count=10,
)
agent = Agent(ap)
print(agent)
params_count = sum(p.numel() for p in agent.parameters())
print(f'{params_count=:_}')
probe_batch = torch.zeros((2, 4, 84, 84))
print(f'{probe_batch.shape=}')
r = agent(probe_batch)
print(f'{r.action.shape=}, {r.log_action_probs.shape=}, {r.probs_entropy.shape=}, {r.value.shape=}')

Agent(
  (network): Sequential(
    (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=512, bias=True)
    (8): ReLU()
  )
  (actor): Linear(in_features=512, out_features=10, bias=True)
  (critic): Linear(in_features=512, out_features=1, bias=True)
)
params_count=1_689_771
probe_batch.shape=torch.Size([2, 4, 84, 84])
r.action.shape=torch.Size([2]), r.log_action_probs.shape=torch.Size([2]), r.probs_entropy.shape=torch.Size([2]), r.value.shape=torch.Size([2, 1])


## Create

In [18]:
ap = Agent.Params(
    actions_count = LS.envs.action_space.n.item()
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)

## capture_video_of_test_rollout

In [19]:
def capture_video_of_test_rollout(agent, max_steps=10_000):
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S") # generate unique dir name in order to shut up RecordVideo from complaining
    video_dir_name = os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}')
    env = create_test_env(HP.test_env_id, video_dir_name=video_dir_name)
    obs, _ = env.reset(seed=HP.random_seed)
    game_meta = {}
    
    with eval_guard(agent):
        with torch.no_grad():
            for step in range(0, max_steps): 
                obs = torch.tensor(einops.rearrange(obs, 'f h w -> 1 f h w')).to(CONFIG.cuda_device)
                action = agent(obs).action.cpu().numpy()[0]
                obs, reward, terminated, truncated, info = env.step(action)
                
                if (terminated or truncated) and env.get_wrapper_attr('was_real_done'):
                    break

    game_meta['reward'] = env.get_wrapper_attr('episode_returns')
    game_meta['frames_count'] = env.get_wrapper_attr('episode_lengths')
    game_meta['steps_count'] = step + 1
    
    env.close() # this forces video recording to complete and write video file
    
    video_fnames = list(filter(lambda fn: os.path.isfile(os.path.join(video_dir_name, fn)), os.listdir(video_dir_name)))
    assert len(video_fnames) == 1, len(video_fnames)
    video_fname = os.path.join(video_dir_name, video_fnames[0])
    video_meta = {}
    
    with open(video_fname, 'rb') as f:
        container = av.open(f)
        video_meta['fps'] = float(container.streams.video[0].average_rate)
        video_stream = container.streams.video[0]
        video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
        
    return video_fname, dict(game=game_meta, video=video_meta)

In [20]:
# @launchit.disable
capture_video_of_test_rollout(LS.agent)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


('/home/misha/dev/mine/neurolab/run/17_rl/video-17a_ppo_atari_envpool_01-launch0-2026.04.28-11:57:13/rl-video-episode-0.mp4',
 {'game': {'reward': 2.0, 'frames_count': 836, 'steps_count': 205},
  'video': {'fps': 30.0, 'duration': 27.9}})

# TRAIN

## Configure

In [21]:
# @launchit.disable
# @launchit.collect

# Training params
HP.rollouts_count = 1 
HP.steps_count = 128 # the number of steps to run in each environment per policy rollout
HP.minibatches_count = 4
HP.update_epochs = 4 # the K epochs to update the policy
HP.gamma = 0.99 # the discount factor gamma
HP.gae_lambda = 0.95 # the lambda for the general advantage estimation
HP.optimizer = 'Adam(eps=0.00001)'
HP.learn_rate: str = '0.00025,linear()'
HP.capture_video = 'every(10)'

# PPO related
HP.clip_coef = 0.1 # the surrogate clipping coefficient
HP.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
HP.ent_coef = 0.01 # coefficient of the entropy
HP.vf_coef = 0.5 # coefficient of the value function
HP.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
HP.target_kl = None # e target KL divergence threshold
HP.norm_adv = True # Toggles advantages normalization

# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'env_id': 'Breakout-v5',
 'envs_count': 8,
 'test_env_id': 'BreakoutNoFrameskip-v4',
 'separate_networks': False,
 'rollouts_count': 1,
 'steps_count': 128,
 'minibatches_count': 4,
 'update_epochs': 4,
 'optimizer': 'Adam(eps=0.00001)',
 'learn_rate': '0.00025,linear()',
 'capture_video': 'every(10)',
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'max_grad_norm': 0.5,
 'target_kl': None,
 'norm_adv': True}


## Create

In [22]:
batch_size = int(HP.envs_count * HP.steps_count)
minibatch_size = int(batch_size // HP.minibatches_count)
LOG(f'{batch_size=}, {minibatch_size=}')

ump = hp_parse_universal_module(HP.optimizer)
assert not ump.args
lr_params = hp_parse_learn_rate(HP.learn_rate)
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), lr=lr_params.learn_rate, **ump.kwargs)
lr_scheduler = LrSchedulerWrapper(optimizer, lr_params, HP.rollouts_count)
capture_video_policy = LS.create_capture_video_policy(HP.capture_video)

batch_size=1024, minibatch_size=256


## Train

In [23]:
# A-la scoreboard for current play round (run num_steps within each of envs)
obs = torch.zeros((HP.steps_count, HP.envs_count) + LS.envs.single_observation_space.shape).to(CONFIG.cuda_device)
actions = torch.zeros((HP.steps_count, HP.envs_count) + LS.envs.single_action_space.shape).to(CONFIG.cuda_device)
logprobs = torch.zeros((HP.steps_count, HP.envs_count)).to(CONFIG.cuda_device)
rewards = torch.zeros((HP.steps_count, HP.envs_count)).to(CONFIG.cuda_device)
dones = torch.zeros((HP.steps_count, HP.envs_count)).to(CONFIG.cuda_device)
values = torch.zeros((HP.steps_count, HP.envs_count)).to(CONFIG.cuda_device)
last_episode_rewards = np.zeros(HP.envs_count)
last_episode_lengths = np.zeros(HP.envs_count)
LOG(f'{obs.shape=}, {actions.shape=}, {logprobs.shape=}, {rewards.shape=}, {dones.shape=}, {values.shape=}')

obs.shape=torch.Size([128, 8, 4, 84, 84]), actions.shape=torch.Size([128, 8]), logprobs.shape=torch.Size([128, 8]), rewards.shape=torch.Size([128, 8]), dones.shape=torch.Size([128, 8]), values.shape=torch.Size([128, 8])


In [27]:
global_step = 0
start_time = time.time()
next_obs, _ = LS.envs.reset()
next_obs = torch.Tensor(next_obs).to(CONFIG.cuda_device)
next_done = torch.zeros(HP.envs_count).to(CONFIG.cuda_device)

for rollout_serial in tqdm(range(0, HP.rollouts_count + 1)): 
    for step in range(0, HP.steps_count): # exec rollout within each env
        global_step += HP.envs_count
        obs[step] = next_obs
        dones[step] = next_done

        # ALGO LOGIC: action logic
        with torch.no_grad():
            action, logprob, _, value = LS.agent(next_obs)
            
        values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        # TRY NOT TO MODIFY: execute the game and log data.
        next_obs, reward, terminations, truncations, infos = LS.envs.step(action.cpu().numpy())
        next_obs = torch.Tensor(next_obs).to(CONFIG.cuda_device, non_blocking=True) # heavy thing, non_blocking may speed up a little
        next_done = np.logical_or(terminations, truncations)
        next_done = torch.Tensor(next_done).to(CONFIG.cuda_device)
        rewards[step] = torch.tensor(reward).to(CONFIG.cuda_device).view(-1)

        for idx, d in enumerate(next_done):
            if d and infos["lives"][idx] == 0:
                last_episode_rewards[idx] = infos['r'][idx]
                last_episode_lengths[idx] = infos['l'][idx]

    advantages = torch.zeros_like(rewards).to(CONFIG.cuda_device) # [num_steps, num_envs]
    
    with torch.no_grad():
        next_value = LS.agent.get_value(next_obs).reshape(1, -1)
        lastgaelam = 0
        
        for t in reversed(range(HP.steps_count)):
            if t == HP.steps_count - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
                
            delta = rewards[t] + HP.gamma * nextvalues * nextnonterminal - values[t]
            lastgaelam = delta + HP.gamma * HP.gae_lambda * nextnonterminal * lastgaelam
            advantages[t] = lastgaelam
            
        returns = advantages + values

    b_obs = obs.reshape((-1,) + LS.envs.single_observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + LS.envs.single_action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    # Optimizing the policy and value network
    data_loader = DataLoader(np.arange(batch_size), shuffle=True, batch_size=minibatch_size)
    clipfracs = []
    
    for epoch in range(HP.update_epochs):
        for mb_inds in data_loader:
            _, newlogprob, entropy, newvalue = LS.agent(b_obs[mb_inds], b_actions.long()[mb_inds])
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                # calculate approx_kl http://joschu.net/blog/kl-approx.html
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > HP.clip_coef).float().mean().item()]

            mb_advantages = b_advantages[mb_inds]
            
            if HP.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            # Policy loss ("pg" stands for policy gradient)
            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - HP.clip_coef, 1 + HP.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            # Value loss
            newvalue = newvalue.view(-1)
            
            if HP.clip_vloss:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -HP.clip_coef,
                    HP.clip_coef,
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            entropy_loss = entropy.mean()
            loss = pg_loss - HP.ent_coef * entropy_loss + v_loss * HP.vf_coef

            if rollout_serial > 0:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(LS.agent.parameters(), HP.max_grad_norm)
                optimizer.step()

        if HP.target_kl is not None and approx_kl > HP.target_kl:
            break

    if rollout_serial > 0:
        lr_scheduler.step(rollout_serial)
    
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    # TRY NOT TO MODIFY: record rewards for plotting purposes
    summary_writer.add_scalar("charts/sps", int(global_step / (time.time() - start_time)), global_step) # steps per second
    summary_writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
    summary_writer.add_scalar("charts/episodic_return", last_episode_rewards.mean(), global_step)
    summary_writer.add_scalar("charts/episodic_length", last_episode_lengths.mean(), global_step)
    
    summary_writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
    summary_writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
    summary_writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
    summary_writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
    summary_writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
    summary_writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
    summary_writer.add_scalar("losses/explained_variance", explained_var, global_step)

    if capture_video_policy(rollout_serial):
        video_fname, video_meta = capture_video_of_test_rollout(LS.agent)
        _, video_fname_ext = os.path.splitext(video_fname)
        ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
        remote_video_fname = f'{ts}-{rollout_serial:06}.{video_fname_ext.lstrip('.')}'
        summary_writer.add_file(video_fname, remote_video_fname)
        summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
        ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
        summary_writer.add_text('videos', ref_text, global_step)

    summary_writer.flush()

  0%|          | 0/2 [00:00<?, ?it/s]

## Save

In [ ]:
# @launchit.disable_2
artifact_registry = new_artifact_registry()
lc = HP.launch_component()

with io.BytesIO() as b:
    torch.save(LS.agent.state_dict(), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)

with io.StringIO() as b:
    json.dump(dataclasses.asdict(LS.agent.params), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

# LaunchIt!

## TRAIN

In [28]:
# @launchit.disable
launchit_t0 = time.time()

In [29]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    artifact_registry_obj = new_artifact_registry(is_real=True)
    artifact_registry_obj.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=component_version, expandvars=expandvars, collect_inds=[1], disable_inds=[1])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=1
Creating /home/misha/dev/mine/neurolab/17_rl/17a_ppo_atari_envpool_01-launch1.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17a_ppo_atari_envpool_01-launch1.ipynb"


## Optuna (model selection)

### Templates

In [45]:
# @launchit.disable
# @launchit.collect_3
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP = Hyperparameters()
            HP.random_seed = 42
            assert False
        case _:
            assert False, f'Unsupported {study_serial=}'            

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 1
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")